# MySQL Create Trigger


MySQL triggers are a powerful feature that allows you to automatically execute a specified action in response to certain events on a particular table. They are useful for enforcing business rules, validating input data, maintaining audit trails, and automatically updating related tables.

Here's a detailed explanation of MySQL triggers, complete with examples to help you understand their usage easily.

## What is a Trigger?

A **trigger** is a database object that is automatically executed or fired when certain events occur on a table. Triggers can be activated before or after an `INSERT`, `UPDATE`, or `DELETE` operation on a table. The syntax for creating a trigger involves specifying the table, the event type, and whether the trigger should fire before or after the event.

### Syntax

```sql
CREATE TRIGGER trigger_name
{BEFORE | AFTER} {INSERT | UPDATE | DELETE}
ON table_name FOR EACH ROW
BEGIN
    -- SQL statements to be executed
END;
```

### Key Components

- **`trigger_name`**: The name of the trigger.
- **`BEFORE` or `AFTER`**: Specifies whether the trigger should be executed before or after the event.
- **`INSERT`, `UPDATE`, `DELETE`**: Specifies the event that activates the trigger.
- **`table_name`**: The table on which the trigger is defined.
- **`FOR EACH ROW`**: Indicates that the trigger will execute for each row affected by the triggering event.
- **`BEGIN ... END`**: The block where the SQL statements are defined.

## Examples of MySQL Triggers

Let's go through some examples to illustrate how triggers work.

### Example 1: BEFORE INSERT Trigger

Suppose we have a table called `employees` and we want to ensure that every employee's salary is at least $30,000 before inserting a new record. We'll create a `BEFORE INSERT` trigger to enforce this rule.

#### Table Structure

```sql
CREATE TABLE employees (
    emp_id INT AUTO_INCREMENT PRIMARY KEY,
    emp_name VARCHAR(100),
    salary DECIMAL(10, 2)
);
```

#### Create Trigger

```sql
DELIMITER $$

CREATE TRIGGER before_insert_employees
BEFORE INSERT ON employees
FOR EACH ROW
BEGIN
    IF NEW.salary < 30000 THEN
        SET NEW.salary = 30000;
    END IF;
END$$

DELIMITER ;
```

- **`DELIMITER $$`**: Changes the statement delimiter temporarily to `$$` to allow the definition of a trigger without terminating it prematurely with a semicolon (`;`).
- **`NEW`**: A special keyword used to reference the new row to be inserted into the table.
- **Condition**: Checks if the `NEW.salary` is less than $30,000. If it is, it sets the `NEW.salary` to $30,000.

#### Inserting Records

```sql
INSERT INTO employees (emp_name, salary) VALUES ('Alice', 25000);
INSERT INTO employees (emp_name, salary) VALUES ('Bob', 35000);
```

#### Result

```sql
SELECT * FROM employees;
```

| emp_id | emp_name | salary |
|--------|----------|--------|
| 1      | Alice    | 30000  |
| 2      | Bob      | 35000  |

- The trigger automatically adjusts Alice's salary to $30,000 before inserting the record.

### Example 2: AFTER UPDATE Trigger

Now let's create a trigger that logs every salary change into a separate table called `salary_changes` whenever an employee's salary is updated.

#### Table Structure for Logging

```sql
CREATE TABLE salary_changes (
    change_id INT AUTO_INCREMENT PRIMARY KEY,
    emp_id INT,
    old_salary DECIMAL(10, 2),
    new_salary DECIMAL(10, 2),
    change_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
```

#### Create Trigger

```sql
DELIMITER $$

CREATE TRIGGER after_update_employees
AFTER UPDATE ON employees
FOR EACH ROW
BEGIN
    IF OLD.salary != NEW.salary THEN
        INSERT INTO salary_changes (emp_id, old_salary, new_salary) 
        VALUES (OLD.emp_id, OLD.salary, NEW.salary);
    END IF;
END$$

DELIMITER ;
```

- **`OLD`**: A special keyword used to reference the row before the update.
- **`AFTER UPDATE`**: Ensures that the trigger fires after the salary has been updated.
- **Condition**: Checks if the `OLD.salary` is different from the `NEW.salary`. If true, it logs the change in the `salary_changes` table.

#### Updating Records

```sql
UPDATE employees SET salary = 32000 WHERE emp_name = 'Alice';
UPDATE employees SET salary = 36000 WHERE emp_name = 'Bob';
```

#### Result

```sql
SELECT * FROM salary_changes;
```

| change_id | emp_id | old_salary | new_salary | change_date         |
|-----------|--------|------------|------------|---------------------|
| 1         | 1      | 30000      | 32000      | 2024-08-01 10:45:12 |
| 2         | 2      | 35000      | 36000      | 2024-08-01 10:46:15 |

- The trigger logs each salary change into the `salary_changes` table with the `old_salary`, `new_salary`, and `change_date`.

### Example 3: BEFORE DELETE Trigger

Consider a scenario where we want to prevent the deletion of an employee record if their salary is above a certain threshold, say $50,000. We'll use a `BEFORE DELETE` trigger to enforce this restriction.

#### Create Trigger

```sql
DELIMITER $$

CREATE TRIGGER before_delete_employees
BEFORE DELETE ON employees
FOR EACH ROW
BEGIN
    IF OLD.salary > 50000 THEN
        SIGNAL SQLSTATE '45000' 
        SET MESSAGE_TEXT = 'Cannot delete employee with salary above 50,000';
    END IF;
END$$

DELIMITER ;
```

- **`SIGNAL SQLSTATE '45000'`**: Generates a user-defined error to prevent the deletion operation.
- **Condition**: Checks if the `OLD.salary` is greater than $50,000. If it is, the trigger prevents the deletion by raising an error.

#### Deleting Records

```sql
DELETE FROM employees WHERE emp_name = 'Bob'; -- Fails if Bob's salary is > 50000
```

#### Expected Outcome

If Bob's salary is above $50,000, the delete operation fails with an error message:

```
ERROR 1644 (45000): Cannot delete employee with salary above 50,000
```

### Example 4: AFTER INSERT Trigger with Multiple Actions

Let's create a trigger that not only updates another table but also sends a notification upon inserting a new record into the `employees` table.

#### Notification Table

```sql
CREATE TABLE notifications (
    notification_id INT AUTO_INCREMENT PRIMARY KEY,
    message VARCHAR(255),
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
```

#### Create Trigger

```sql
DELIMITER $$

CREATE TRIGGER after_insert_employees
AFTER INSERT ON employees
FOR EACH ROW
BEGIN
    -- Update a summary table
    UPDATE employee_summary
    SET total_employees = total_employees + 1;
    
    -- Insert a notification
    INSERT INTO notifications (message)
    VALUES (CONCAT('New employee added: ', NEW.emp_name, ', Salary: ', NEW.salary));
END$$

DELIMITER ;
```

- **Multiple Actions**: The trigger performs two actions: updates the `employee_summary` table and inserts a notification.
- **`CONCAT`**: Used to concatenate strings for the notification message.

#### Inserting Records

```sql
INSERT INTO employees (emp_name, salary) VALUES ('Charlie', 28000);
```

#### Result

```sql
SELECT * FROM notifications;
```

| notification_id | message                            | created_at          |
|-----------------|------------------------------------|---------------------|
| 1               | New employee added: Charlie, Salary: 30000 | 2024-08-01 11:00:12 |

- The trigger logs a notification every time a new employee is added.

### Use Cases for Triggers

Triggers are often used in the following scenarios:

1. **Enforcing Business Rules**:
   - Automatically ensure data consistency and integrity.
   - Example: Prevent negative inventory by checking stock levels before an update.

2. **Auditing and Logging**:
   - Maintain audit trails for data changes.
   - Example: Log changes to sensitive data for compliance purposes.

3. **Validation**:
   - Validate data before it's inserted or updated.
   - Example: Check input data meets certain criteria before allowing a transaction.

4. **Automatic Calculations**:
   - Perform automatic calculations or updates.
   - Example: Automatically update totals or averages in summary tables.

5. **Notifications and Alerts**:
   - Send notifications or alerts based on specific data changes.
   - Example: Trigger an email alert if a high-value transaction occurs.

### Important Considerations

- **Performance Impact**: Triggers can affect performance, especially if they contain complex logic or interact with multiple tables. Use them judiciously.
- **Debugging**: Debugging triggers can be challenging because they execute automatically. Ensure thorough testing.
- **Security**: Consider the security implications of triggers, especially if they modify data or perform sensitive operations.

### Summary

MySQL triggers are a robust tool for automating database operations in response to specific events. By using triggers, you can enforce business rules, maintain audit trails, and automate calculations. Understanding the different types of triggers (`BEFORE` and `AFTER`) and how to use them effectively is